In [ ]:
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

from os.path import dirname, abspath
from sys import path

SCRIPT_DIR = dirname(abspath(__name__))
path.append(dirname(SCRIPT_DIR))

from utils.browser import Linkedin0, URLs
import pandas as pd

In [ ]:
browser = Linkedin0(URLs.JOBS)

def details_of(job, xpath:str) -> str:
    return job.find_element(By.XPATH, xpath).text.replace("\n", "").replace("  ", "")

In [ ]:
jobs, page = [], 1
while True:

    print(page, end=" ")
    for job in browser.wait.until(EC.visibility_of_all_elements_located((By.XPATH, "*//ul/li/div/div/div/div[2]"))):

        jobs.append({"last_status": details_of(job, "div[2]"),
            **{key: details_of(job, f"div/div[{k+1}]") for k, key in enumerate(["position", "company", "location"])}})

    if not (btn := browser.find_element(By.XPATH, "//button[@aria-label='Next']")).is_enabled(): break
    browser.execute_script("arguments[0].click();", btn)
    page += 1

In [ ]:
browser.logout()

In [ ]:
pd.DataFrame(jobs).to_csv("data/raw.csv", index=False, sep=";")